# Heart Failure Clinical Records: Exploratory Analysis & Risk Stratification

***Author:** Mohamed Osman  
**Domain:** Biomedical Engineering / Health Data Science  
**Tools:** Python, Pandas, NumPy  

---

### Project Overview
This notebook focuses on **Clinical Feature Engineering** and **Risk Stratification** using heart failure patient data. 
Key objectives:
1. Normalize and cast data types for medical precision.
2. Derive clinical indicators (e.g., **HFrEF**, **Renal Impairment**, **Cardiorenal Syndrome Risk**).
3. Aggregate mortality rates using multi-dimensional pivot tables (`pd.pivot_table`).

Import tools and Create DataFrame

In [5]:
import os

import numpy as np
import pandas as pd

# 1. Open data file and Create DataFrame
BASE_DIR = os.getcwd()
FILE_PATH = os.path.join(BASE_DIR, 'heart_failure_clinical_records.csv')

df_heart_fcr = pd.read_csv(FILE_PATH)
df_heart_fcr.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,55.0,0,748,0,45,0,263358.03,1.3,137,1,1,88,0
1,65.0,0,56,0,25,0,305000.00,5.0,130,1,0,207,0
2,45.0,0,582,1,38,0,319000.00,0.9,140,0,0,244,0
3,60.0,1,754,1,40,1,328000.00,1.2,126,1,0,90,0
4,95.0,1,582,0,30,0,461000.00,2.0,132,1,0,50,1


## 1. Data Inspection & Type Conversion
We verify dataset schema, check for missing values, and cast binary indicators to `bool` and continuous metrics to explicit float/int data types.

In [6]:
# Check data summary & missing values
print("--- Missing Values Check ---")
print(df_heart_fcr.isna().sum())

print("\n--- Summary Statistics ---")
df_heart_fcr.describe()

--- Missing Values Check ---
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

--- Summary Statistics ---


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,60.288736,0.474400,586.760600,0.439400,37.734600,0.364800,265075.404370,1.369106,136.808200,0.645600,0.311800,130.678800,0.313600
std,11.697243,0.499394,976.733979,0.496364,11.514855,0.481422,97999.758622,1.009750,4.464236,0.478379,0.463275,77.325928,0.464002
min,40.000000,0.000000,23.000000,0.000000,14.000000,0.000000,25100.000000,0.500000,113.000000,0.000000,0.000000,4.000000,0.000000
25%,50.000000,0.000000,121.000000,0.000000,30.000000,0.000000,215000.000000,0.900000,134.000000,0.000000,0.000000,74.000000,0.000000
50%,60.000000,0.000000,248.000000,0.000000,38.000000,0.000000,263358.030000,1.100000,137.000000,1.000000,0.000000,113.000000,0.000000
75%,68.000000,1.000000,582.000000,1.000000,45.000000,1.000000,310000.000000,1.400000,140.000000,1.000000,1.000000,201.000000,1.000000
max,95.000000,1.000000,7861.000000,1.000000,80.000000,1.000000,850000.000000,9.400000,148.000000,1.000000,1.000000,285.000000,1.000000


In [8]:
# Convert Data Types for Clinical Precision
df_heart_fcr = df_heart_fcr.astype({
    'anaemia': bool,
    'DEATH_EVENT': bool,
    'diabetes': bool,
    'high_blood_pressure': bool,
    'smoking': bool,
    'ejection_fraction': np.int64,
    'time': np.int64,
    'sex': np.int64,
    'age': np.float64,
    'creatinine_phosphokinase': np.float64,
    'serum_creatinine': np.float64,
    'serum_sodium': np.float64,
    'platelets': np.float64
})

# Overall Mortality Baseline
baseline_mortality = df_heart_fcr['DEATH_EVENT'].value_counts(normalize=True) * 100
print(f"Overall Dataset Mortality Rate:\n{baseline_mortality.to_string()}")

Overall Dataset Mortality Rate:
DEATH_EVENT
False    68.64
True     31.36


## 2. Clinical Feature Engineering
Creating domain-specific diagnostic markers based on medical guidelines:
* **EF Category:** Classifying Ejection Fraction into **HFrEF** ($EF < 40\%$), **HFmrEF** ($40\% \le EF < 50\%$), and **Preserved** ($EF \ge 50\%$).
* **Renal Impairment:** Gender-aware creatinine thresholds ($>1.2$ mg/dL for females, $>1.4$ mg/dL for males).
* **Cardiorenal Syndrome Risk:** Co-occurrence of reduced EF ($<40\%$) and elevated Serum Creatinine ($>1.5$ mg/dL).

In [9]:
# 1. Unified Heart Failure Classification (EF)
ef_conditions = [
    df_heart_fcr['ejection_fraction'] < 40,
    (df_heart_fcr['ejection_fraction'] >= 40) & (df_heart_fcr['ejection_fraction'] < 50)
]
ef_choices = ['HFrEF', 'HFmrEF']
df_heart_fcr['EF_Category'] = np.select(ef_conditions, ef_choices, default='Preserved')

# 2. Severe LV Dysfunction
df_heart_fcr['Severe_Left_Ventricular_Dysfunction'] = np.where(
    df_heart_fcr['ejection_fraction'] < 30, 'Severe_LV_Dysfunction', 'Normal'
)

# 3. Severe Myocardial Injury
df_heart_fcr['Severe_Myocardial_Injury'] = np.where(
    df_heart_fcr['creatinine_phosphokinase'] > 1000, 'Severe_Injury', 'Normal'
)

# 4. Renal Impairment (Gender-Specific)
df_heart_fcr['Renal_Impairment'] = np.where(
    ((df_heart_fcr['serum_creatinine'] > 1.2) & (df_heart_fcr['sex'] == 0)) |
    ((df_heart_fcr['serum_creatinine'] > 1.4) & (df_heart_fcr['sex'] == 1)),
    'Renal_Impairment', 'Normal'
)

# 5. Severe Renal Failure
df_heart_fcr['Severe_Renal_Failure'] = np.where(
    df_heart_fcr['serum_creatinine'] > 2.5, 'Severe_Failure', 'Normal'
)

# 6. Electrolyte & Platelet Disturbances
df_heart_fcr['Hyponatremia'] = np.where(df_heart_fcr['serum_sodium'] < 135, 'Hyponatremia', 'Normal')
df_heart_fcr['Hypernatremia'] = np.where(df_heart_fcr['serum_sodium'] > 145, 'Hypernatremia', 'Normal')
df_heart_fcr['Thrombocytopenia'] = np.where(df_heart_fcr['platelets'] < 150000, 'Thrombocytopenia', 'Normal')
df_heart_fcr['Thrombocytosis'] = np.where(df_heart_fcr['platelets'] > 450000, 'Thrombocytosis', 'Normal')

# 7. Complex Syndromes & Risk Triads
df_heart_fcr['Cardiorenal_Syndrome_Risk'] = np.where(
    (df_heart_fcr['ejection_fraction'] < 40) & (df_heart_fcr['serum_creatinine'] > 1.5),
    'High', 'Low'
)

df_heart_fcr['Vascular_Metabolic_Risk_Triad'] = np.where(
    (df_heart_fcr['diabetes']) & (df_heart_fcr['high_blood_pressure']) & (df_heart_fcr['smoking']),
    'High', 'Low'
)

## 3. Mortality Aggregations & Clinical Cross-Tabulation
Evaluating mortality rates across custom risk categories using grouped aggregations and cross-tabulation tables.

In [12]:
# Cardiorenal Syndrome Mortality Summary
cardiorenal_summary = df_heart_fcr.groupby('Cardiorenal_Syndrome_Risk')['DEATH_EVENT'].agg(['count', 'mean'])
cardiorenal_summary['Mortality_Percentage'] = cardiorenal_summary['mean'] * 100
cardiorenal_summary

,count,mean,Mortality_Percentage
Cardiorenal_Syndrome_Risk,,,
High,848,0.707547,70.754717
Low,4152,0.233141,23.314066


In [13]:
# Cross-Tabulation: Renal Impairment vs Ejection Fraction Category (Mortality Rate %)
pivot_risk = pd.pivot_table(
    df_heart_fcr,
    values='DEATH_EVENT',
    index='Renal_Impairment',
    columns='EF_Category',
    aggfunc='mean'
)

# Render as percentage table
pivot_risk * 100

EF_Category,HFmrEF,HFrEF,Preserved
Renal_Impairment,,,
Normal,5.396476,28.282330,17.391304
Renal_Impairment,57.792208,66.735113,45.512821


## 4. Export Processed Dataset
Save the dataset with newly engineered features to a clean CSV file for subsequent visualization and modeling stages.

In [18]:
# Save processed DataFrame
PROCESSED_PATH = 'processed_heart_failure_records.csv'
df_heart_fcr.to_csv(PROCESSED_PATH, index=False)
print(f" Processed dataset successfully exported to: {PROCESSED_PATH}")

 Processed dataset successfully exported to: processed_heart_failure_records.csv


### 📋 Processed Cohort Preview (20 Representative Cases)
Below is a sample of 20 patient records illustrating the engineered clinical features (`EF_Category`, `Renal_Impairment`, and `Cardiorenal_Syndrome_Risk`) mapped directly alongside baseline clinical parameters.

In [17]:
# Select core clinical columns for clean horizontal visualization
display_columns = [
    'age', 
    'sex', 
    'ejection_fraction', 
    'serum_creatinine', 
    'EF_Category', 
    'Renal_Impairment', 
    'Cardiorenal_Syndrome_Risk', 
    'DEATH_EVENT'
]

# Display a reproducible random sample of 20 patients
df_heart_fcr[display_columns].sample(20, random_state=42)

,age,sex,ejection_fraction,serum_creatinine,EF_Category,Renal_Impairment,Cardiorenal_Syndrome_Risk,DEATH_EVENT
1501,65.0,0,35,0.8,HFrEF,Normal,Low,False
2586,72.0,1,25,1.0,HFrEF,Normal,Low,True
2653,70.0,0,60,1.3,Preserved,Renal_Impairment,Low,True
1055,68.0,1,25,1.0,HFrEF,Normal,Low,True
705,70.0,1,35,1.1,HFrEF,Normal,Low,False
106,65.0,1,35,1.1,HFrEF,Normal,Low,False
589,70.0,0,38,1.1,HFrEF,Normal,Low,False
2468,40.0,1,30,0.9,HFrEF,Normal,Low,False
2413,55.0,0,38,1.3,HFrEF,Renal_Impairment,Low,False
1600,43.0,0,50,1.3,Preserved,Renal_Impairment,Low,False
